In [1]:
import cv2
import pickle

video_path = 'people.mp4'
cap = cv2.VideoCapture(video_path)
success, frame = cap.read()
cap.release()

line_points = []

def draw_line(event, x, y, flags, param):
    global line_points
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(line_points) < 2:
            line_points.append((x, y))

    if event == cv2.EVENT_RBUTTONDOWN:
        line_points = []

cv2.namedWindow("Draw Line")
cv2.setMouseCallback("Draw Line", draw_line)

while True:
    img_display = frame.copy()

    for pt in line_points:
        cv2.circle(img_display, pt, 5, (0, 0, 255), -1)

    if len(line_points) == 2:
        cv2.line(img_display, line_points[0], line_points[1], (0, 255, 0), 2)

    cv2.imshow("Draw Line", img_display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        if len(line_points) == 2:
            with open('line_config.pkl', 'wb') as f:
                pickle.dump(line_points, f)
            print("Line Saved to line_config.pkl")
        break

cv2.destroyAllWindows()


Line Saved to line_config.pkl


In [2]:
import cv2
import pickle
import numpy as np
from ultralytics import YOLO

# get model
model = YOLO('yolo11n.pt')

# get coordinates of line
with open('line_config.pkl', 'rb') as f:
    line_pts = pickle.load(f)

video_path = 'people.mp4'
cap = cv2.VideoCapture(video_path)

track_history = {} # id, [x,y]

counter_list = []

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # tracking
    results = model.track(frame, persist=True, classes=[0], device='cpu', verbose=False)[0]

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.int().cpu().tolist()
        track_ids = results.boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2 # Center of each person

            #get previous position
            previous_position = track_history.get(track_id)
            track_history[track_id] = (cx, cy)

            if previous_position is not None:
                # get Y of previous position
                prev_y = previous_position[1]

                lx1, ly1 = line_pts[0]
                lx2, ly2 = line_pts[1]
                line_y = ly1

                # compare only Y coordinates
                if prev_y < line_y and cy >= line_y: # from top to bottom
                    if track_id not in counter_list:
                        counter_list.append(track_id)

                elif prev_y > line_y and cy <= line_y: # from bottom to top
                    if track_id not in counter_list:
                        counter_list.append(track_id)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    pt1, pt2 = line_pts[0], line_pts[1]
    cv2.line(frame, pt1, pt2, (0, 0, 255), 3)
    cv2.putText(frame, f"Total Count: {len(counter_list)}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

    cv2.imshow("People Counter", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()